In [1]:
import pandas as pd
#%pip install nsepy- NSE stock exchange data
from nsepy import get_history as gh
#pip install --upgrade statsmodels
import datetime as dt
#%pip install yfinance
import yfinance as yf
import numpy as np

In [2]:
import yfinance as yf

start = "2021-07-01"
end = "2022-01-29"

skt_data = yf.download(
    "TATACONSUM.NS",
    start=start,
    end=end
)

skt_data.head()
skt_data=skt_data[["Open","High","Low","Close"]]
#skt_data

[*********************100%***********************]  1 of 1 completed


In [3]:
#column="Close"
from sklearn.preprocessing import MinMaxScaler
Ms=MinMaxScaler()
data1=Ms.fit_transform(skt_data)
print("Len:",data1.shape)

Len: (145, 4)


In [4]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])
data1

,Open,High,Low,Close
0,0.337436,0.281337,0.370188,0.338841
1,0.348206,0.330363,0.385983,0.378442
2,0.412564,0.345404,0.444225,0.392604
3,0.400000,0.331476,0.409674,0.373196
4,0.353846,0.315042,0.376357,0.375557
...,...,...,...,...
140,0.158974,0.186351,0.182626,0.184107
141,0.179488,0.086351,0.066634,0.000000
142,0.000000,0.002507,0.000000,0.085497
143,0.051282,0.000000,0.061698,0.076055


In [5]:
training_size=round(len(data1)*.80)
X_train=data1[:training_size]#eg. Zeor to 79
print("The training set lenght is:",len(X_train))
X_test=data1[training_size:]#eg. 80 to 99
print("The test set lenght is:",len(X_test))
total=(len(X_test)+len(X_train))-len(data1)
print("The remaining if any pending to split is:",total)
Y_train=data1[:training_size]
print("The training set lenght is:",len(Y_train))
Y_test=data1[training_size:]
print("The test set lenght is:",len(Y_test))
total=(len(Y_test)+len(Y_train))-len(data1)
print("The remaining if any pending to split is:",total)

The training set lenght is: 116
The test set lenght is: 29
The remaining if any pending to split is: 0
The training set lenght is: 116
The test set lenght is: 29
The remaining if any pending to split is: 0


In [6]:
import warnings#built in python library
warnings.filterwarnings("ignore")

In [7]:
performance={"Model":[],"RMSE":[],"Mape":[],"Lag":[],"Test":[]}
performance

{'Model': [], 'RMSE': [], 'Mape': [], 'Lag': [], 'Test': []}

In [8]:
def combination(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]
    test_obs=28
    train=datasetTwo[:-test_obs]# it means all rows : except last 28 rows.
    test=datasetTwo[-test_obs:]# only last 28 rows.
    from statsmodels.tsa.api import VAR
    for i in[1,2,3,4,5,6,7,8,9,10]:
        model=VAR(train)
        results=model.fit(i)
        print('order=',i)
        print('AIC=',results.aic)
        print('BIC=',results.bic)
        print()
    x=model.select_order(maxlags=12)# it means how many days i required to lag the data.
    order=x.selected_orders["aic"]#automatically selectes the best lags out of 12.
    result=model.fit(order) # based on best lags, it will crete model.
    lagged_Values=train.values[-order:]
    pred=result.forecast(y=lagged_Values,steps=28)#The steps belongs to the no.of days to forecast.
    preds=pd.DataFrame(pred,columns=listt)
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    from sklearn.metrics import mean_squared_error
    rmse = round(np.sqrt(mean_squared_error(test, pred)), 2)
    from sklearn.metrics import mean_absolute_percentage_error
    mape=mean_absolute_percentage_error(test,pred)
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["Mape"].append(mape)
    performance["Lag"].append(order)
    performance["Test"].append(test_obs)
    perf=pd.DataFrame(performance)
    return perf,result,pred
    
    
    

In [9]:
listt=["Close","High","Open","Low"]

In [15]:
perf,result,pred=combination(data1,listt)

['Close', 'High', 'Open', 'Low']
order= 1
AIC= -28.436359887420465
BIC= -27.961602957919368

order= 2
AIC= -28.433618373282727
BIC= -27.574335272229884

order= 3
AIC= -28.239548178729308
BIC= -26.99145765840901

order= 4
AIC= -28.076091876400877
BIC= -26.434831950096104

order= 5
AIC= -28.013819564819435
BIC= -25.974945411348116

order= 6
AIC= -27.843020692603375
BIC= -25.402002493222895

order= 7
AIC= -27.759498158897344
BIC= -24.911718864061704

order= 8
AIC= -27.69939088817984
BIC= -24.440143911535372

order= 9
AIC= -27.56659989860818
BIC= -23.891086735512026

order= 10
AIC= -27.512890334024046
BIC= -23.416218101764677



In [11]:
data1

,Open,High,Low,Close
0,0.337436,0.281337,0.370188,0.338841
1,0.348206,0.330363,0.385983,0.378442
2,0.412564,0.345404,0.444225,0.392604
3,0.400000,0.331476,0.409674,0.373196
4,0.353846,0.315042,0.376357,0.375557
...,...,...,...,...
140,0.158974,0.186351,0.182626,0.184107
141,0.179488,0.086351,0.066634,0.000000
142,0.000000,0.002507,0.000000,0.085497
143,0.051282,0.000000,0.061698,0.076055


In [12]:
perf

,Model,RMSE,Mape,Lag,Test
0,"[Close, High, Open, Low]",0.1,3.941311e+13,1,28


In [16]:
preds=pd.DataFrame(pred,columns=listt)
preds

,Close,High,Open,Low
0,0.106394,0.049583,0.097357,0.132031
1,0.112325,0.064955,0.118511,0.138759
2,0.119356,0.071413,0.123069,0.144313
3,0.126158,0.078870,0.130136,0.150973
4,0.132843,0.086089,0.136814,0.157334
5,0.139409,0.093183,0.143389,0.163600
6,0.145856,0.100150,0.149845,0.169751
7,0.152186,0.106992,0.156185,0.175792
8,0.158403,0.113710,0.162411,0.181724
9,0.164508,0.120307,0.168524,0.187549
